# OncoVault — CBC Leukemia Risk Assessment Model (`leukemia-cbc-v1`)
### Google Colab Training & Multi-Site Evaluation Notebook

- **Training Cohort:** 3 Multi-Site Training Datasets ($N=26,922$ subjects)
- **Validation Cohort:** 7 Independent Validation Sites ($N=361,260$ subjects)
- **Independent Test Cohort:** True-World Test Site ($N=58,481$ subjects)

> **Clinical Note:** Uses exactly the 9 finalized OncoVault CBC features (`WBC`, `RBC`, `HGB`, `PLT`, `NEUT#`, `NEUT%`, `MONO#`, `RDW-SD`, `RDW-CV`).

In [ ]:
# 1. Setup Environment & Imports
import os, csv, glob, json, joblib
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
print('Environment initialized.')

In [ ]:
# 2. Load 9 Finalized CBC Features
CBC_COLS = ['WBC(10^9/L)', 'RBC(10^12/L)', 'HGB(g/L)', 'PLT(10^9/L)', 'NEUT#(10^9/L)', 'NEUT%(%)', 'MONO#(10^9/L)', 'RDW-SD(fL)', 'RDW-CV(%)']

def load_cbc_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)
        col_indices = [header.index(c) for c in CBC_COLS]
        label_idx = header.index('leukemia_label')
        X, y = [], []
        for r in reader:
            if not r or not any(cell.strip() for cell in r): continue
            X.append([float(r[i].strip()) for i in col_indices])
            y.append(1 if r[label_idx].strip().lower() in ['leukemia', '1', 'positive'] else 0)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

train_files = sorted(glob.glob('oncovault_ai/data/cbc/training/*.csv'))
X_train_raw = np.vstack([load_cbc_file(fp)[0] for fp in train_files])
y_train = np.concatenate([load_cbc_file(fp)[1] for fp in train_files])
print(f'Training records across {len(train_files)} sites: {len(X_train_raw):,}')

In [ ]:
# 3. Preprocessing Scaler (Fitted ONLY on Training Sites)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)

# Train Final Gradient Boosting Model
cbc_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
cbc_model.fit(X_train, y_train)
print('CBC Model training complete.')

In [ ]:
# 4. Evaluate on Independent Test Site
X_test_raw, y_test = load_cbc_file('oncovault_ai/data/cbc/test/test_site_true_world.csv')
X_test = scaler.transform(X_test_raw)
preds = cbc_model.predict(X_test)
probs = cbc_model.predict_proba(X_test)[:, 1]
cm = confusion_matrix(y_test, preds)
tn, fp, fn, tp = cm.ravel()

print('=== TRUE-WORLD INDEPENDENT TEST RESULTS ===')
print(f'Accuracy:             {accuracy_score(y_test, preds):.4f}')
print(f'Recall / Sensitivity: {recall_score(y_test, preds):.4f}')
print(f'Specificity:          {tn/(tn+fp):.4f}')
print(f'Precision:            {precision_score(y_test, preds):.4f}')
print(f'F1-Score:             {f1_score(y_test, preds):.4f}')
print(f'ROC-AUC:              {roc_auc_score(y_test, probs):.4f}')
print(f'Confusion Matrix:     TN={tn}, FP={fp}, FN={fn}, TP={tp}')